In [2]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import classification_report
import time
import warnings
warnings.filterwarnings('ignore')

# Seed — final raporda zorunlu, her çalıştırmada aynı sonuç
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz     : {device}")
print(f"GPU       : {torch.cuda.get_device_name(0)}")
print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch   : {torch.__version__}")

Cihaz     : cuda
GPU       : Tesla T4
VRAM      : 15.6 GB
PyTorch   : 2.10.0+cu128


In [4]:
df = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/raw/data.csv', sep=";")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/bdm_proje/data/raw/data.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


After mounting your Google Drive, you'll need to specify the correct path to your `data.csv` file. You can typically find it under `/content/drive/MyDrive/` followed by the folder structure where you saved it. For example, if it's directly in your 'MyDrive', the path would be `'/content/drive/MyDrive/data.csv'`.

In [ ]:
# Update this path to the correct location of your data.csv file in Google Drive
# For example, if it's in a folder named 'turkish-slm-benchmark' within your Drive, it might be:
# data_path = '/content/drive/MyDrive/turkish-slm-benchmark/data/raw/data.csv'
# To find the exact path, you can use `!ls /content/drive/MyDrive/` in a new cell.
data_path = '/content/drive/MyDrive/bdm_proje/data/raw/data.csv' # <<< Change this line to your actual path

df = pd.read_csv(data_path, sep=";")

In [1]:
df.head(20)

NameError: name 'df' is not defined

In [ ]:
from sklearn.model_selection import train_test_split

LABEL_NAMES = {
    0: "Yardım Talebi",
    1: "Kayıp Bildirimi",
    2: "Altyapı Hasarı",
    3: "Bağış/Koordinasyon",
    4: "Diğer/İlgisiz"
}

print(f"Toplam veri : {len(df)} tweet")
print(f"\nEtiket dağılımı:")
for k, v in LABEL_NAMES.items():
    n = (df['label_id'] == k).sum()
    print(f"  {k} - {v:<22}: {n}")

# ADIM 1: %10 test ayır → geriye %90 kalır
train_val, test_df = train_test_split(
    df,
    test_size   = 0.10,        # %10 test
    random_state= SEED,
    stratify    = df['label_id']
)

# ADIM 2: kalan %90'ın %11.1'i → toplam %10 val olur
# Neden 0.111? → 0.90 * 0.111 ≈ 0.10
train_df, val_df = train_test_split(
    train_val,
    test_size   = 0.111,       # %90'ın %11.1'i = toplam %10 val
    random_state= SEED,
    stratify    = train_val['label_id']
)

print(f"\nBölme tamamlandı:")
print(f"  Train : {len(train_df)}  (%{len(train_df)/len(df)*100:.0f})")
print(f"  Val   : {len(val_df)}   (%{len(val_df)/len(df)*100:.0f})")
print(f"  Test  : {len(test_df)}   (%{len(test_df)/len(df)*100:.0f})")

print(f"\nTest seti etiket dağılımı:")
print(test_df['label_id'].value_counts().sort_index())

# Kaydet
train_df.to_csv('/content/drive/MyDrive/bdm_proje/data/processed/train.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/bdm_proje/data/processed/val.csv',   index=False)
test_df.to_csv('/content/drive/MyDrive/bdm_proje/data/processed/test.csv',  index=False)

print("\n✅ Kaydedildi!")

Toplam veri : 793 tweet

Etiket dağılımı:
  0 - Yardım Talebi         : 289
  1 - Kayıp Bildirimi       : 98
  2 - Altyapı Hasarı        : 100
  3 - Bağış/Koordinasyon    : 105
  4 - Diğer/İlgisiz         : 201

Bölme tamamlandı:
  Train : 633  (%80)
  Val   : 80   (%10)
  Test  : 80   (%10)

Test seti etiket dağılımı:
label_id
0    29
1    10
2    10
3    11
4    20
Name: count, dtype: int64

✅ Kaydedildi!


In [ ]:
#İLK DEFA MODELİ İNDİRİRKEN KULLANILACAK KOD


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Model yükleniyor: {MODEL_ID}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,     # torch_dtype yerine dtype
    device_map="auto"        # GPU varsa otomatik kullan
)

model.eval()

# Model bilgileri
param_count = sum(p.numel() for p in model.parameters())

model_size_mb = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
) / 1e6

print("✅ Model yüklendi!")
print(f"Parametre sayısı : {param_count/1e6:.0f}M")
print(f"Model boyutu     : {model_size_mb:.0f} MB")

if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"VRAM kullanımı   : {torch.cuda.memory_allocated()/1e9:.2f} GB")




Model yükleniyor: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model yüklendi!
Parametre sayısı : 1100M
Model boyutu     : 2200 MB
GPU              : Tesla T4
VRAM kullanımı   : 2.20 GB


In [ ]:
# İNDİRİLEN MODELİ KAYDETME KISMI

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Ensure these paths match your desired save location in Google Drive
tokenizer.save_pretrained("/content/drive/MyDrive/bdm_proje/models/tinyllama")
model.save_pretrained("/content/drive/MyDrive/bdm_proje/models/tinyllama")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

MODEL_PATH = "/content/drive/MyDrive/bdm_proje/models/tinyllama"

if not os.path.exists(MODEL_PATH):
    print(f"❌ HATA: {MODEL_PATH} dizini bulunamadı! Lütfen Drive yolunu kontrol edin.")
else:
    print(f"🔄 Local model yükleniyor: {MODEL_PATH}")

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    # Model - T4 GPU için optimize edilmiş yükleme
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

    # Pad token eksikse ayarla
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id

    model.eval()

    print("✅ Model başarıyla yüklendi!")
    if torch.cuda.is_available():
        print(f"🚀 GPU Aktif: {torch.cuda.get_device_name(0)}")
        print(f"📊 VRAM Kullanımı: {torch.cuda.memory_allocated()/1e9:.2f} GB")

🔄 Local model yükleniyor: /content/drive/MyDrive/bdm_proje/models/tinyllama


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model başarıyla yüklendi!
🚀 GPU Aktif: Tesla T4
📊 VRAM Kullanımı: 2.20 GB


In [ ]:
def zero_shot_siniflandir(tweet, model, tokenizer, max_new_tokens=40):
    # Modelin yanıt vermesini kolaylaştırmak için daha doğrudan bir prompt
    prompt = f"Classify the tweet into one category: Help Request, Missing Person, Infrastructure, Donation, Other.\n\nTweet: {tweet}\nCategory:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_return_sequences=1,
            # EOS token'ı görmezden gelerek modelin bir şeyler yazmasını zorla
            eos_token_id=None if tokenizer.eos_token_id is None else tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return response, 0

def parse_label(label_text):
    label_text = label_text.lower()
    if "help" in label_text: return 0
    if "person" in label_text or "missing" in label_text: return 1
    if "infra" in label_text or "damage" in label_text: return 2
    if "donat" in label_text or "coord" in label_text: return 3
    return 4 # Varsayılan olarak Diğer

# Tekil test
test_res, _ = zero_shot_siniflandir("Enkaz altında sesler var acil yardım.", model, tokenizer)
print(f"Örnek Model Yanıtı: {repr(test_res)}")
print(f"Tahmin Edilen ID: {parse_label(test_res)}")

Örnek Model Yanıtı: 'Help Request\n\nTweet: İnsanlık yerinde sesler var.\nCategory: Missing Person\n\nTweet: İnsanlık y'
Tahmin Edilen ID: 0


In [1]:
from tqdm import tqdm

print(f"Zero-shot başladı...: {len(test_df)} tweet, model: TinyLlama-1.1B\n")

sonuclar = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    tweet      = row['tweet']
    gercek     = row['label_id']

    yanit, sure = zero_shot_siniflandir(tweet, model, tokenizer)
    tahmin      = parse_label(yanit)
    dogru_mu    = (tahmin == gercek)

    # Her adımda çıktıyı yazdır
    print(f"\n[{idx}] Tweet: {tweet[:80]}...")
    print(f"    ✅ Gerçek : {LABEL_NAMES[gercek]} ({gercek})")
    print(f"    ℹ️ Yanıt  : {repr(yanit)}")
    print(f"    ⌛ Tahmin : {LABEL_NAMES[tahmin]} ({tahmin})")
    print(f"    🎯 Doğru mu: {'EVET ✅' if dogru_mu else 'HAYIR ❌'}")
    print("-" * 30)

    sonuclar.append({
        'tweet'       : tweet,
        'gercek'      : gercek,
        'gercek_etiket' : LABEL_NAMES[gercek],
        'model_yanit' : yanit,
        'tahmin'      : tahmin,
        'dogru_mu'    : dogru_mu,
        'sure_ms'     : sure
    })

df_sonuc = pd.DataFrame(sonuclar)

print(f"\n✅ Tamamlandı!")
print(f"   Parse edilemeyen : {(df_sonuc['tahmin'] == -1).sum()} tweet")
print(f"   Ortalama süre    : {df_sonuc['sure_ms'].mean():.1f} ms/tweet")

NameError: name 'test_df' is not defined

In [ ]:
from sklearn.metrics import (classification_report,
                              confusion_matrix,
                              accuracy_score,
                              f1_score)

# Parse edilemeyen tahminleri filtrele
df_gecerli = df_sonuc[df_sonuc['tahmin'] != -1].copy()

print(f"Geçerli tahmin : {len(df_gecerli)} / {len(df_sonuc)}")
print(f"Parse edilemedi: {len(df_sonuc) - len(df_gecerli)}\n")

y_true = df_gecerli['gercek'].tolist()
y_pred = df_gecerli['tahmin'].tolist()

# Ana metrikler
accuracy  = accuracy_score(y_true, y_pred)
f1_macro  = f1_score(y_true, y_pred, average='macro',  zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("=" * 50)
print("TinyLlama-1.1B — ZERO-SHOT SONUÇLARI")
print("=" * 50)
print(f"Accuracy      : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"F1 Macro      : {f1_macro:.4f}")
print(f"F1 Weighted   : {f1_weighted:.4f}")
print(f"Inference hızı: {df_sonuc['sure_ms'].mean():.1f} ms/tweet")
print()
print("Sınıf bazlı rapor:")
print(classification_report(
    y_true, y_pred,
    target_names=[LABEL_NAMES[i] for i in sorted(LABEL_NAMES.keys())],
    zero_division=0
))

Geçerli tahmin : 80 / 80
Parse edilemedi: 0

TinyLlama-1.1B — ZERO-SHOT SONUÇLARI
Accuracy      : 0.3625  (36.2%)
F1 Macro      : 0.1064
F1 Weighted   : 0.1929
Inference hızı: 0.0 ms/tweet

Sınıf bazlı rapor:
                    precision    recall  f1-score   support

     Yardım Talebi       0.36      1.00      0.53        29
   Kayıp Bildirimi       0.00      0.00      0.00        10
    Altyapı Hasarı       0.00      0.00      0.00        10
Bağış/Koordinasyon       0.00      0.00      0.00        11
     Diğer/İlgisiz       0.00      0.00      0.00        20

          accuracy                           0.36        80
         macro avg       0.07      0.20      0.11        80
      weighted avg       0.13      0.36      0.19        80



In [ ]:
import json
import os

# Sonuç dizinini oluştur
output_dir = '/content/drive/MyDrive/bdm_proje/results/zero_shot'
os.makedirs(output_dir, exist_ok=True)

ozet = {
    "model": "TinyLlama-1.1B-Chat-v1.0",
    "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "parametre": "1.1B",
    "yontem": "zero-shot",
    "test_size": len(test_df),
    "gecerli_tahmin": len(df_sonuc[df_sonuc['tahmin'] != -1]),
    "parse_edilemedi": len(df_sonuc[df_sonuc['tahmin'] == -1]),
    "inference_ms": round(df_sonuc['sure_ms'].mean(), 1),
    "model_boyutu_mb": round(model_size_mb, 0)
}

file_path = os.path.join(output_dir, 'tinyllama_zero_shot_ozet.json')
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(ozet, f, ensure_ascii=False, indent=2)

print(f"✅ {file_path} adresine kaydedildi!")

✅ /content/drive/MyDrive/bdm_proje/results/zero_shot/tinyllama_zero_shot_ozet.json adresine kaydedildi!
